# 04.01 · Tokenization

**Chapter:** Transformers & How LLMs Work · **Milestone:** M1

## Goals
- [ ] Why models use subword tokens (not characters or words)
- [ ] Implement Byte-Pair Encoding (BPE) from scratch: train, encode, decode
- [ ] Use real tokenizers: `tiktoken` (OpenAI) and Hugging Face `tokenizers` (open models)
- [ ] Explain how tokens drive cost, context limits and odd failures (spelling, arithmetic, non-English text)

## Resources
- [Karpathy – Let's build the GPT Tokenizer](https://www.youtube.com/watch?v=zduSFxRajkE) — free
- [Tiktokenizer (visual playground)](https://tiktokenizer.vercel.app/) — free
- [HF LLM Course – Tokenizers](https://huggingface.co/learn/llm-course/chapter6/1) — free

Models don't see text; they see integer IDs. The tokenizer is a separate, fixed component trained before the model.

In [ ]:
from collections import Counter

## 1. Characters vs words vs subwords
- **Characters**: tiny vocabulary, but sequences get very long (attention cost grows with length²).
- **Words**: short sequences, but huge vocabulary and every unseen word is "unknown".
- **Subwords (BPE)**: frequent words become one token, rare words split into pieces. Works for any string. Everyone uses this.

Modern BPE works on **UTF-8 bytes**, so any text (emoji, code, any language) is representable with a 256-symbol base.

In [ ]:
text = "café 🤖"
print(list(text), "->", len(text), "characters")
print(list(text.encode("utf-8")), "->", len(text.encode("utf-8")), "bytes")

## 2. BPE from scratch
Start with bytes. Repeatedly find the most frequent adjacent pair and merge it into a new token. Record merges in order.

In [ ]:
def get_pair_counts(ids):
    return Counter(zip(ids, ids[1:]))


def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i + 1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out


def train_bpe(text, vocab_size):
    ids = list(text.encode("utf-8"))
    merges = {}  # (a, b) -> new id
    for new_id in range(256, vocab_size):
        counts = get_pair_counts(ids)
        if not counts:
            break
        pair = max(counts, key=counts.get)
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id
    return merges


def encode(text, merges):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        counts = get_pair_counts(ids)
        pair = min(counts, key=lambda p: merges.get(p, float("inf")))  # earliest-learned merge first
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids


def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), new_id in merges.items():  # dicts keep insertion order = merge order
        vocab[new_id] = vocab[a] + vocab[b]
    return vocab


def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

In [ ]:
corpus = (
    "The tokenizer is trained on a corpus. The more often a pair of tokens appears, the earlier it is merged. "
    "Tokenization matters: tokens determine cost, context length and what the model can easily see. "
) * 20
merges = train_bpe(corpus, vocab_size=256 + 60)
vocab = build_vocab(merges)

sample = "The tokens determine the cost."
ids = encode(sample, merges)
print(f"{len(sample.encode())} bytes -> {len(ids)} tokens")
print([vocab[i].decode("utf-8", errors="replace") for i in ids])
assert decode(ids, vocab) == sample
print("learned merges (first 10):", [vocab[i].decode() for i in list(merges.values())[:10]])

Real tokenizers add a **pre-tokenization** regex (split on spaces/punctuation/digits first, so merges never cross word
boundaries) and **special tokens** like `<|endoftext|>` or chat-role markers.

## 3. Real tokenizers
- **tiktoken**: OpenAI's fast BPE. `o200k_base` is the encoding of recent OpenAI models.
- **Hugging Face tokenizers**: what open models (Llama, Qwen, Gemma, Mistral…) ship with; loaded via `AutoTokenizer`.
Different models = different tokenizers = different token counts for the same text.

In [ ]:
import tiktoken
from transformers import AutoTokenizer

enc = tiktoken.get_encoding("o200k_base")
gpt2 = tiktoken.get_encoding("gpt2")  # 2019, 50k vocab, mostly English
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")  # downloads only small tokenizer files

s = "Tokenization is surprisingly important!"
t_ids = enc.encode(s)
print("tiktoken :", len(t_ids), [enc.decode([i]) for i in t_ids])
q_ids = qwen.encode(s)
print("Qwen3    :", len(q_ids), qwen.convert_ids_to_tokens(q_ids))
print("vocab sizes:", enc.n_vocab, len(qwen))

## 4. Why tokens explain weird behaviour
- **Languages**: English-centric tokenizers need more tokens for other languages → higher cost, less fits in context.
  Modern ~200k-vocabulary tokenizers closed the gap for big languages (Spanish, German); lower-resource scripts still pay more.
- **Spelling**: the model sees `strawberry` as 1–3 opaque chunks, not letters. Counting 'r's is genuinely hard for it.
- **Numbers**: digits are chunked inconsistently, which hurts arithmetic.
- **Whitespace/code**: indentation and spaces are tokens too.

In [ ]:
samples = {
    "English": "The weather is nice today and I want to go for a walk in the park.",
    "Spanish": "El clima está agradable hoy y quiero salir a caminar por el parque.",
    "German": "Das Wetter ist heute schön und ich möchte im Park spazieren gehen.",
    "Japanese": "今日は天気が良いので、公園を散歩したいです。",
    "Hindi": "आज मौसम अच्छा है और मैं पार्क में टहलना चाहता हूँ।",
    "Python code": "def add(a, b):\n    return a + b\n",
}
for name, s in samples.items():
    print(f"{name:12} chars={len(s):3}  gpt2={len(gpt2.encode(s)):3}  o200k={len(enc.encode(s)):3}  qwen3={len(qwen.encode(s)):3}")

for word in ["strawberry", " strawberry", "12345678", "1234 + 5678"]:
    print(repr(word), "->", [enc.decode([i]) for i in enc.encode(word)])

## 5. Tokens = money and context
APIs bill per token and limit context in tokens. Rule of thumb for English: **1 token ≈ 4 characters ≈ 0.75 words**.
Always count with the model's own tokenizer when it matters (chapter 05.05).

## Exercises
1. Train your BPE with 500 merges on a few pages of your own writing (or `data/tinyshakespeare.txt` after 04.03). Inspect the longest tokens.
2. Compare token counts for the same paragraph in English and your native language, with both tokenizers. What's the cost ratio?
3. Add GPT-style pre-tokenization to your BPE: split with `regex` pattern `r"'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+"`
   (needs `uv add regex`) and merge within chunks only.

In [ ]:
# your code here